# Automated LLM Red-Teaming & Defense Analysis

This notebook provides exploratory analysis, visualization figures, and statistical validation of empirical campaign data extracted from `data/redteam.db`.

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

conn = sqlite3.connect('../data/redteam.db')

# Load full attempts record
df = pd.read_sql_query('SELECT * FROM attempts', conn)
print(f'Total records logged: {len(df)}')
df.head()

## 1. Vulnerability Rate: Strategy vs Target Model (Guard OFF)

In [ ]:
baseline_df = df[df['guard_active'] == 0]

grouped = baseline_df.groupby(['strategy', 'target_model']).agg(
    total=('id', 'count'),
    violations=('judge_violated', 'sum')
).reset_index()
grouped['success_rate_pct'] = (100.0 * grouped['violations'] / grouped['total']).round(1)

pivot = grouped.pivot(index='strategy', columns='target_model', values='success_rate_pct').fillna(0)

plt.figure(figsize=(8, 5))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='Reds', vmin=0, vmax=100)
plt.title('Vulnerability Rate (%) Across Strategies & Models')
plt.ylabel('Strategy')
plt.xlabel('Target Model')
plt.tight_layout()
plt.show()

## 2. Attacker Efficiency (Attempts to First Breach)

In [ ]:
breaches = df[(df['judge_violated'] == 1) & (df['guard_blocked'] == 0)]

plt.figure(figsize=(9, 4))
sns.histplot(data=breaches, x='attempt_number', hue='strategy', multiple='dodge', bins=10, shrink=0.8)
plt.title('Attacker Iteration Number at Violation')
plt.xlabel('Attempt Number (PAIR Iteration)')
plt.ylabel('Violation Count')
plt.tight_layout()
plt.show()

## 3. Defense Guard Effectiveness (Before vs After)

In [ ]:
guard_comp = df.groupby(['guard_active', 'target_model']).agg(
    total=('id', 'count'),
    violations=('judge_violated', lambda x: ((x == 1) & (df.loc[x.index, 'guard_blocked'] == 0)).sum()),
    blocked=('guard_blocked', 'sum')
).reset_index()
guard_comp['effective_success_rate'] = (100.0 * guard_comp['violations'] / guard_comp['total']).round(1)
guard_comp